# Session 1 (Mon Jul 20) — RAG, end to end

You've already built a chatbot with a web UI, given it tools, and put a pod on the cluster.
The one piece that never really landed is **RAG** — and it's the piece the whole project is
named after. Week 5 was self-study; today we do it together, properly, in one hour.

By the end of this notebook the bot answers a real NRP question using the **real NRP
documentation**, with citations.

1. **Why we need RAG at all** — watch the model confidently make something up
2. **Embeddings** — text becomes numbers, and similar meaning lands nearby
3. **Chunking** — why we cut docs into pieces
4. **The index** — storing chunks so we can search them
5. **`search()`** — find the five chunks that matter
6. **The prompt** — hand those chunks to the model and demand citations
7. **`answer_question()`** — the one function the app and the evaluation both call

> Same `.env` as always. Run every cell as we go — don't just watch.

## Setup

```bash
pip install "openai==1.55.0" "httpx<0.28" python-dotenv chromadb
```

> The `httpx<0.28` pin is not optional: the newest httpx dropped an argument this version of
> `openai` still passes, and without the pin the client raises `unexpected keyword argument
> 'proxies'` the moment you create it.

**If a model ever hangs or errors mid-session**, it's usually the shared cluster, not your
code. `client.models.list()` shows what's currently served — swap `CHAT_MODEL` to another
one and carry on. We use `gpt-oss` because it's consistently the quickest to respond.

In [ ]:
import os, json, glob, textwrap
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(
    api_key=os.environ["NRP_LLM_TOKEN"],
    base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"),
    timeout=120,
)

CHAT_MODEL = "gpt-oss"          # what writes the answers
EMBED_MODEL = "qwen3-embedding" # what turns text into numbers

def ask(prompt, model=CHAT_MODEL):
    r = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print("client ready ->", client.base_url)

---
## A. Why we need RAG

### A1. Ask the model something only NRP's docs know

The model has never read NRP's documentation. But it was trained to be *helpful*, and a
helpful-sounding wrong answer is the most dangerous thing an LLM does.

In [ ]:
q = "In the NRP Nautilus cluster, what do I put in a pod spec to request an A100 GPU?"
print(ask(q))

**Look carefully at what came back.** It's fluent, it's confident, it's shaped exactly like
a correct answer — and the details are invented. It cannot know NRP's specific conventions.

This is the whole problem. Two ways to fix it:

| Approach | What it means | Why not / why yes |
|---|---|---|
| Fine-tuning | Retrain the model on NRP docs | Expensive, slow, and needs redoing every time docs change |
| **RAG** | **Look up the docs, paste them into the prompt** | **Cheap, instant, and updates the moment docs do** |

RAG is almost embarrassingly simple: **find the right documents first, then let the model
read them.** Everything below is just doing that well.

### A2. Prove it — paste the answer in by hand

Before building any machinery, let's confirm the idea works.

In [ ]:
real_doc = """
To request a GPU on Nautilus, add a resource limit to your container spec:
  resources:
    limits:
      nvidia.com/gpu: 1
To select a specific GPU type, use a nodeSelector on nvidia.com/gpu.product,
for example: nvidia.com/gpu.product: NVIDIA-A100-SXM4-80GB
Always set both requests and limits for cpu and memory.
"""

grounded = f"""Answer the question using ONLY the documentation below.
If the documentation does not contain the answer, say so.

DOCUMENTATION:
{real_doc}

QUESTION: {q}"""

print(ask(grounded))

Same model, same question — now correct, because the answer was **in the prompt**.

Everything else in this notebook exists to answer one question automatically:
*which documents do we paste in?*

---
## B. Embeddings — how a computer compares meaning

To find the right docs we need to compare a question to thousands of chunks. Keyword
matching breaks immediately: someone asks *"how do I get a graphics card"* and the doc says
*"request a GPU"* — zero words in common, same meaning.

An **embedding** turns text into a long list of numbers (a vector) positioned so that text
with similar *meaning* gets similar numbers. Then "related" becomes measurable arithmetic.

### B1. Embed some text and look at it

In [ ]:
def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=[text]).data[0].embedding

v = embed("How do I request a GPU pod?")
print("dimensions:", len(v))
print("first 8 numbers:", [round(x, 4) for x in v[:8]])

That list of numbers *is* the sentence, as far as the math is concerned.

### B2. Measure similarity

**Cosine similarity** scores two vectors from -1 (opposite) to 1 (identical). Watch how it
tracks meaning, not shared words.

In [ ]:
import math

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb)

question = embed("How do I request a GPU pod?")
candidates = {
    "To request a GPU, add nvidia.com/gpu to resource limits.": None,
    "Attach a graphics card to your Kubernetes workload.":      None,
    "The cafeteria serves lunch until 2pm.":                    None,
}
for text in candidates:
    print(f"{cosine(question, embed(text)):+.3f}   {text}")

The second line shares **almost no words** with the question but scores high — that's
semantic search working. The lunch line scores low. That's the entire retrieval engine.

> **The one rule that will bite you:** you must use the *same* embedding model to index your
> documents and to search them. Two models put text in two different coordinate systems, and
> comparing across them returns pure noise. That's why we wrap it in one `embed()` function
> and always call that.

---
## C. Chunking — cutting docs into searchable pieces

We don't embed whole pages. Two reasons:

- A 4,000-word page is *about* twenty things. Its single vector is a blurry average of all
  of them, and matches nothing precisely.
- We paste retrieved text into the prompt. Whole pages blow the context window fast.

So we cut pages into **chunks** of roughly 500 tokens (~2,000 characters), with a small
overlap so a sentence spanning a boundary isn't lost.

### C1. The real NRP documentation

We've already collected the NRP user documentation for you: **85 pages of real markdown**,
sitting in the `nrp-docs/` folder next to this notebook. They were pulled straight from
nrp.ai, and each file keeps its **source URL at the top** — that line is what lets the bot
cite where an answer came from.

Nothing to download. Just point at the folder.

In [ ]:
DOCS = "nrp-docs"          # bundled next to this notebook — real NRP userdocs
paths = glob.glob(f"{DOCS}/**/*.md", recursive=True)
print("markdown files found:", len(paths))
print("example:", paths[0] if paths else "(is the nrp-docs/ folder next to this notebook?)")

### C2. Chunk them

Simple, readable, and good enough. The chunk dictionary is our agreed schema — keep
`source_url`, because that's what makes citations possible.

In [ ]:
def chunk_text(text, size=2000, overlap=200):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size])
        i += size - overlap
    return out

chunks = []
for path in paths:
    raw = open(path, encoding="utf-8", errors="ignore").read()
    if len(raw.strip()) < 200:          # skip near-empty stubs
        continue
    rel = os.path.relpath(path, DOCS).replace(".md", "")
    for n, piece in enumerate(chunk_text(raw)):
        chunks.append({
            "id": f"{rel.replace('/', '_')}__{n:03d}",
            "source_url": f"https://nrp.ai/documentation/{rel}/",
            "title": rel.split("/")[-1].replace("-", " ").title(),
            "text": piece,
        })

print("total chunks:", len(chunks))
print(json.dumps(chunks[0], indent=2)[:500])

> 📊 **Student 1 — this is your poster panel.** The number of files, the number of chunks,
> and that example chunk are Figures 1, 2 and 4. Screenshot this output now.

---
## D. The index — store the chunks so we can search them

A **vector database** stores each chunk with its embedding and answers "what's closest to
this?" quickly. We use Chroma because it runs from a local folder with no server.

### D1. Build the index

We embed every chunk once and hand Chroma the vectors. The trick that keeps this fast: send
the embedding model a **whole batch of chunks per API call** instead of one at a time. The
full corpus indexes in a few seconds.

In [ ]:
import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)  # quiet noise
import chromadb

coll = (chromadb.PersistentClient(path="./chroma_db")
        .get_or_create_collection("nrp_docs"))

def embed_batch(texts):                      # one API call for many texts — much faster
    return [d.embedding for d in
            client.embeddings.create(model=EMBED_MODEL, input=texts).data]

B = 32
for i in range(0, len(chunks), B):
    batch = chunks[i:i + B]
    coll.upsert(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embed_batch([c["text"] for c in batch]),
        metadatas=[{"source_url": c["source_url"], "title": c["title"]} for c in batch],
    )
    print(f"indexed {min(i + B, len(chunks))}/{len(chunks)}")

print("collection size:", coll.count())

`upsert` (rather than `add`) makes this safe to re-run — same id overwrites instead of
erroring. Re-running a half-finished ingest is something you will do a lot.

---
## E. `search()` — the retrieval contract

One function, one job: question in, most relevant chunks out. Everything downstream depends
only on this shape.

In [ ]:
def search(query, k=5):
    res = coll.query(query_embeddings=[embed(query)], n_results=k)
    return [
        {"text": d, "source_url": m["source_url"], "title": m["title"], "score": s}
        for d, m, s in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

for hit in search("How do I request a GPU?", k=3):
    print(f"[{hit['score']:.3f}] {hit['title']}  <- {hit['source_url']}")
    print(textwrap.shorten(hit["text"].replace("\n", " "), 160), "\n")

**Read the retrieved text before moving on.** If the right document isn't in this list, no
prompt can save the answer — the information simply isn't there.

This is *the* debugging habit for RAG: when an answer is wrong, look at what was retrieved
**first**. If retrieval missed, fix chunking or `k`. If retrieval was fine, fix the prompt.

> 📊 **Student 2 — your poster panel.** This output is Figure 3.

---
## F. The prompt — grounding and citations

Now we do automatically what we did by hand in A2.

In [ ]:
def build_prompt(query, hits):
    context = "\n\n---\n\n".join(
        f"[Source: {h['title']} | {h['source_url']}]\n{h['text']}" for h in hits)
    return f"""You are a helpful assistant for users of the National Research Platform (NRP).
Answer the question using ONLY the documentation below.
If the documentation does not contain the answer, say so honestly — do not guess.
Cite the source of each fact you use.

DOCUMENTATION:
{context}

QUESTION: {query}"""

hits = search("How do I request a GPU?", k=5)
print(ask(build_prompt("How do I request a GPU?", hits)))

Three instructions are doing real work in that prompt:

- **"ONLY the documentation below"** — stops the model blending in half-remembered training data
- **"say so honestly"** — gives it an exit that isn't inventing something
- **"cite the source"** — makes the answer checkable by a human

> 📊 **Student 3 — your poster panel.** Run this question through A1 (no retrieval) and
> through here (with retrieval), screenshot both side by side. That's Figure 6, and it is
> the single most persuasive image on the poster.

---
## G. `answer_question()` — the contract

One function the Streamlit app *and* Wednesday's evaluation both call. Because both go
through this, an improvement here shows up in both places at once.

In [ ]:
def answer_question(query, k=5):
    hits = search(query, k=k)
    answer = ask(build_prompt(query, hits))
    return {"answer": answer, "chunks": hits}

result = answer_question("How do I run a batch job on Nautilus?")
print(result["answer"])
print("\nSOURCES")
for c in result["chunks"]:
    print(" -", c["title"], "->", c["source_url"])

# The honesty exit — ask something the docs genuinely don't cover:
print("\n" + "=" * 60)
print("HONESTY CHECK — a question the docs can't answer:\n")
print(answer_question("How do I make a giant purple dinosaur costume?")["answer"])

That's RAG, complete. Trace it once more and make sure you can narrate every arrow:

```
question ──► embed ──► search the index ──► top-5 chunks
                                                │
                     build_prompt(question, chunks)
                                                │
                                          NRP LLM
                                                │
                                    answer + citations
```

---
## H. 🔧 Your turn — before Wednesday

**The job: turn this notebook into an `ingest.py` that runs on its own.**

Right now chunking and embedding happen in cells, in order, with you watching. On Wednesday
the *pod* has to do it unattended at startup — and then Streamlit just answers questions
against the finished index. So everyone writes the script version.

### Everyone does this

1. **`ingest.py`** — a single file that: reads every `.md` under `nrp-docs/`, chunks it,
   embeds it in batches, and saves to a Chroma collection on disk. No notebook cells, no
   manual steps: `python ingest.py` and it's done.
2. **Make it safe to re-run.** If the index is already built, print that and exit instead of
   re-embedding. Wednesday's pod restarts, and re-embedding every time is the thing we're
   trying to avoid.
   ```python
   if coll.count() >= len(chunks):
       print("index already built — skipping")
   ```
3. **`app.py`** — a Streamlit app that *only queries*. It opens the existing index, runs
   `search()`, builds the grounded prompt, and shows the answer plus its sources. It must
   **not** chunk or embed the corpus — that work is already finished by the time it starts.

That split — **build once at startup, query forever after** — is exactly how it gets
deployed. Wednesday we put your two files in a ConfigMap and let the cluster run them.

Run both locally before Wednesday:

```bash
python ingest.py          # a few seconds; says "skipping" the second time
streamlit run app.py      # ask it a real NRP question
```

### And your poster piece

| You | Also do this | Poster figure |
|---|---|---|
| **S1** Ingest | Own `ingest.py`. Report pages in / chunks out | Fig 1, 2, 4 |
| **S2** Retrieval | Own `search()`. Compare `k=3` vs `k=10` on real questions | Fig 3 |
| **S3** RAG core | Own `build_prompt()`. Test the honesty exit | Fig 5, 6 |
| **S4** Interface | Own `app.py` — sources expander, spinner, empty-answer handling | Fig 7 |
| **S5** Deploy | Read Session 2's manifests. Start the architecture diagram | Fig 8, 9, 10 |
| **S6** Evaluation | **Write 20 real NRP questions with expected answers.** Needed Wednesday | Fig 11, 12 |
| **S7** The frame | Draft INTRODUCTION. Set up the poster file from the template | header, intro |

**Student 6 — your 20 questions gate Wednesday's second half.** They must exist before the
call. Mix easy factual lookups with a couple the docs genuinely don't cover, so we can check
the bot admits ignorance instead of inventing.

Stuck? Post in `#rehs-2026`. Do not go quiet — that's the only real failure mode left.